# Execute the complete 2.2 selection pipeline

This is the single reproduction entry point. It unconditionally invokes `run_all.py`, which starts a clean rebuild after completion and resumes an interrupted run automatically.

## Full resumable run

The run defaults to four CUDA workers with one internal XGBoost worker per fit. This is conservative relative to the successful six-worker H100 sweep in `derived_8.2-hyperparameters-1.5`; rerun every cell after a VM timeout to resume the first incomplete stage.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.2-feature-selection-2.2"
RUNNER = EXP_DIR / "run_all.py"
assert RUNNER.exists(), RUNNER
command = [sys.executable, str(RUNNER), "--device", "cuda", "--workers", "4"]
print("Full run: long-running; rerun this notebook to resume after a VM timeout.")
print("Command:", " ".join(command))

Full run: long-running; rerun this notebook to resume after a VM timeout.
Command: /scratch/user/u.rp352032/MDR-Project/notebooks/.venv/bin/python3 /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/run_all.py --device cpu --workers 16


## Run and display generated results

The master runner owns cleanup, stage ordering, and resume. Once it finishes, this cell loads every report table from `generate_results.py` and prints each underlying CSV path.

In [2]:
import subprocess

subprocess.run(command, cwd=PROJECT_ROOT, check=True)

sys.path.insert(0, str(EXP_DIR))
from generate_results import (
    build_candidate_ceiling_table,
    build_moe_table,
    build_retrospective_table,
    build_validation_table,
)

tables = {
    "Validation": build_validation_table(),
    "Retrospective": build_retrospective_table(),
    "Candidate diagnostics": build_candidate_ceiling_table(),
    "MoE ablation": build_moe_table(),
}
for title, table in tables.items():
    print()
    print(title)
    display(table)
print("Artifacts:")
for path in sorted((EXP_DIR / "artifacts").rglob("*.csv")):
    print(path)


Validation


,dataset,model,beta,R2,RMSE,ubRMSE,Bias,MAE,Med|Err|,Pearson
0,derived_8.0,2.2_global,0.0,0.830559,0.043252,0.040446,0.015325,0.032695,0.024395,0.923998
2,derived_8.0,2.2_global,0.2,0.831860,0.043086,0.040551,0.014562,0.032353,0.023945,0.923289
1,derived_8.0,hand_mdr_v25,0.0,0.878314,0.036654,0.035234,0.010104,0.028614,0.023821,0.949629
3,derived_8.0,hand_mdr_v25,0.2,0.881914,0.036108,0.034854,0.009430,0.028209,0.023156,0.950057
6,derived_8.2,2.1_c1,0.0,0.675166,0.065423,0.064346,-0.011821,0.043890,0.027542,0.828246
10,derived_8.2,2.1_c1,0.2,0.665885,0.066351,0.064957,-0.013530,0.043902,0.027031,0.824926
7,derived_8.2,2.2_clustering_dynamic_k2_shared_plus_delta,0.0,0.709657,0.061852,0.061519,-0.006409,0.041821,0.027849,0.845592
11,derived_8.2,2.2_clustering_dynamic_k2_shared_plus_delta,0.2,0.704337,0.062416,0.062002,-0.007179,0.041916,0.027133,0.843067
4,derived_8.2,2.2_global,0.0,0.737450,0.058817,0.058709,-0.003570,0.040410,0.026841,0.859995
8,derived_8.2,2.2_global,0.2,0.720102,0.060729,0.060569,-0.004409,0.041260,0.027156,0.850514



Retrospective


,artifact_set,dataset,model,beta,R2,RMSE,ubRMSE,Bias,MAE,Med|Err|,Pearson
0,final,derived_8.0,2.2_global,0.0,0.785888,0.043572,0.043569,-0.000479,0.032626,0.024644,0.887216
1,final,derived_8.0,hand_mdr_v25,0.0,0.824642,0.039432,0.039324,-0.002910,0.028297,0.020522,0.908641
2,final,derived_8.0,2.2_global,0.2,0.773456,0.044819,0.044818,-0.000221,0.033657,0.025744,0.880956
3,final,derived_8.0,hand_mdr_v25,0.2,0.824819,0.039412,0.039329,-0.002556,0.028153,0.020671,0.908710
4,final,derived_8.2,2.2_global,0.0,0.604663,0.066209,0.064303,-0.015774,0.048534,0.037277,0.798555
5,final,derived_8.2,V3,0.0,0.653713,0.061966,0.058243,-0.021156,0.046497,0.035615,0.837064
6,final,derived_8.2,2.1_c1,0.0,0.660484,0.061357,0.059682,-0.014239,0.045131,0.034556,0.824708
7,final,derived_8.2,2.2_clustering_dynamic_k2_shared_plus_delta,0.0,0.620704,0.064852,0.062911,-0.015748,0.047110,0.034518,0.809179
8,final,derived_8.2,2.2_global,0.2,0.610358,0.065731,0.063804,-0.015799,0.048159,0.036400,0.801516
9,final,derived_8.2,V3,0.2,0.637738,0.063379,0.059853,-0.020845,0.047386,0.035466,0.828331



Candidate diagnostics


,artifact_set,dataset,outer_selected_n_features,outer_selected_retrospective_R2,retrospective_ceiling_n_features,retrospective_ceiling_R2
2,crossed_candidates_locked_outer,derived_8.0,100,0.780809,150,0.795770
3,crossed_candidates_locked_outer,derived_8.2,80,0.619727,100,0.675726
0,nested,derived_8.0,100,0.780809,100,0.780809
1,nested,derived_8.2,100,0.675726,100,0.675726
4,progressive_crossed_locked_outer,derived_8.0,100,0.780809,150,0.816924



MoE ablation


,artifact_set,dataset,model,beta,R2,RMSE,ubRMSE,Bias,MAE,Med|Err|,Pearson
19,nested,derived_8.2,2.2_clustering_dynamic_k2_shared_plus_delta,0.0,0.623748,0.064591,0.063285,-0.012924,0.048296,0.036710,0.808755
20,nested,derived_8.2,2.2_clustering_frozen_k2_shared_only,0.0,0.608959,0.065849,0.064769,-0.011875,0.049063,0.037234,0.799016
21,nested,derived_8.2,2.2_clustering_refit_k2_shared_plus_delta,0.0,0.618213,0.065065,0.063774,-0.012894,0.048600,0.037181,0.806331
16,nested,derived_8.2,2.2_global,0.0,0.662752,0.061152,0.059610,-0.013647,0.046254,0.035500,0.828629


Artifacts:
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/crossed_candidates_locked_outer/candidate_diagnostics/global_candidates.csv
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/crossed_candidates_locked_outer/derived_8.0/global/outer_fold_metrics.csv
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/crossed_candidates_locked_outer/derived_8.2/global/outer_fold_metrics.csv
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/crossed_candidates_locked_outer/retrospective_test_eval/metrics_by_station.csv
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/crossed_candidates_locked_outer/retrospective_test_eval/metrics_by_year.csv
/scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifact